# Optimization and Optimal Control: from shooting to iLQR

In the previous notebooks we computed motions *instantaneously* (inverse kinematics) or *simulated* the dynamics forward in time. In this notebook we go one step further: we look for a whole **sequence of torques** that drives a robot to a goal, by solving an **Optimal Control Problem (OCP)**.

Our guinea pig is the classical **cart-pole**: a cart moving on a rail, carrying a pole attached by a free (non-actuated) revolute joint. Only the cart is actuated. The system is *underactuated* (1 actuator, 2 degrees of freedom) and *unstable* around the upright position, which makes it the "hello world" of optimal control.

The notebook is organized as follows:
1. Build the cart-pole model in Pinocchio and visualize it.
2. Write the discrete dynamics $x_{k+1} = f(x_k, u_k)$ using Pinocchio's forward dynamics (ABA).
3. Unroll the dynamics over $N$ steps (*single shooting*) and define a cost.
4. Solve the problem with an off-the-shelf solver (SciPy), first with finite-difference gradients...
5. ... then with exact gradients, computed with Pinocchio's analytical derivatives and the chain rule (*adjoint method*).
6. Exploit the temporal structure of the problem: derive the Riccati recursion and implement **iLQR**.
7. Use the feedback gains given by iLQR to reject perturbations, and close the loop with **Model Predictive Control (MPC)**.

## Set up

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin
from numpy.linalg import norm
from scipy.linalg import solve_discrete_are
from scipy.optimize import approx_fprime, minimize

from tp4.cartpole import add_target, build_cartpole, play_trajectory, plot_trajectory, set_side_view
from utils.meshcat_viewer_wrapper import MeshcatVisualizer

rng = np.random.default_rng(0)  # fixed seed: everybody gets the same numbers

## 1. The cart-pole model

The model is built from scratch in `tp4/cartpole.py` (have a look, it is short!). It is made of two joints:

| Joint | Type | Coordinate | Meaning |
|-------|------|------------|---------|
| `cart` | prismatic along $X$ | $p$ | position of the cart on the rail (m) |
| `pole` | revolute around $Y$ | $\theta$ | angle of the pole (rad) |

The convention is that $\theta = 0$ is the pole pointing **up** (unstable equilibrium) and $\theta = \pi$ is the pole hanging **down** (stable equilibrium). Both joints are Euclidean, so here $n_q = n_v = 2$ and we can write
$$
q = \begin{bmatrix} p \\ \theta \end{bmatrix} \in \mathbb{R}^2, \qquad
v = \dot q = \begin{bmatrix} \dot p \\ \dot \theta \end{bmatrix} \in \mathbb{R}^2 .
$$

In [ ]:
robot = build_cartpole()
model, data = robot.model, robot.data
print(model)
print("Pole mass:", model.inertias[2].mass, "kg,  cart mass:", model.inertias[1].mass, "kg")

In [ ]:
viz = MeshcatVisualizer(robot)
set_side_view(viz)
add_target(viz)  # the green ball marks the goal of the pole tip: cart at the origin, pole up
viz.display(np.array([0.0, 0.0]))

Click on the link above to open the viewer in a separate window, or execute the following cell to embed the viewer here:

In [ ]:
hasattr(viz.viewer, "jupyter_cell") and viz.viewer.jupyter_cell()

You can display any configuration $q = (p, \theta)$:

In [ ]:
for theta in np.linspace(0, 2 * np.pi, 50):
    viz.display(np.array([np.sin(theta), theta]))
    time.sleep(0.03)

We store a few dimensions that will be used everywhere:
- the **state** $x = (q, v) \in \mathbb{R}^{n_x}$ with $n_x = n_q + n_v = 4$;
- the **control** $u \in \mathbb{R}^{n_u}$ with $n_u = 1$: the horizontal force applied on the cart.

The joint torque vector is $\tau = S u$ where $S = \begin{bmatrix} 1 & 0 \end{bmatrix}^\top$ is the *actuation matrix*: the pole joint is not actuated.

In [ ]:
nq, nv = model.nq, model.nv
nx, nu = nq + nv, 1
S = np.array([[1.0], [0.0]])  # actuation matrix: tau = S @ u

## 2. Discrete dynamics

### 2.1 From Lagrange to a discrete step

The equation of motion of the robot is (see the previous notebook)
$$
M(q)\,\dot v + b(q, v) = \tau = S u ,
$$
where $M(q)$ is the (symmetric positive definite) mass matrix and $b(q,v)$ gathers the Coriolis, centrifugal and gravity effects. The **forward dynamics** gives the acceleration
$$
a = \mathrm{FD}(q, v, \tau) = M(q)^{-1}\big(\tau - b(q,v)\big),
$$
and is computed efficiently by the *Articulated Body Algorithm*: `pin.aba(model, data, q, v, tau)`.

We discretize time with a step $\Delta t$ and use the **semi-implicit Euler** scheme (the same as in the simulation notebook): the velocity is updated first, then the configuration is integrated with the *new* velocity:
$$
\begin{aligned}
v^+ &= v + \Delta t\, \mathrm{FD}(q, v, S u), \\
q^+ &= q \oplus \Delta t\, v^+ ,
\end{aligned}
$$
where $\oplus$ is `pin.integrate` (here simply $q + \Delta t\, v^+$, since our configuration space is Euclidean). This defines the **discrete dynamics**
$$
x_{k+1} = f(x_k, u_k), \qquad f : \mathbb{R}^{n_x} \times \mathbb{R}^{n_u} \to \mathbb{R}^{n_x}.
$$

In [ ]:
dt = 0.05  # time step (s)

**Question 1.** Implement the function `step(x, u)` that returns $x^+ = f(x, u)$.

In [ ]:
def step(x, u):
    '''Semi-implicit Euler step of the cart-pole dynamics.

    Args:
        x (np.ndarray): state (q, v), shape (nx,).
        u (np.ndarray): control (force on the cart), shape (nu,).

    Returns:
        np.ndarray: next state x+ = f(x, u), shape (nx,).
    '''
    q, v = x[:nq], x[nq:]
    a = pin.aba(model, data, q, v, S @ u)
    # ------------------------- TODO STUDENT (begin) -------------------------
    # TODO: next velocity and next configuration (semi-implicit Euler)
    v_next = None
    q_next = None
    # -------------------------- TODO STUDENT (end) --------------------------
    return np.concatenate([q_next, v_next])

Let us check it: without any force, the pole released slightly off the upright position should fall, and the cart should move in the opposite direction (the horizontal momentum is conserved).

In [ ]:
x = np.array([0.0, 0.1, 0.0, 0.0])  # pole slightly tilted
X_fall = [x]
for k in range(60):
    x = step(x, np.zeros(nu))
    X_fall.append(x)
X_fall = np.array(X_fall)

play_trajectory(viz, X_fall, dt)

We can also plot the trajectory, with the helper `plot_trajectory` of `tp4/cartpole.py`:

In [ ]:
plot_trajectory(X_fall, dt, title="Free fall")

### 2.2 Rollout

Given an initial state $x_0$ and a sequence of controls $U = (u_0, \dots, u_{N-1})$, the trajectory $X = (x_0, \dots, x_N)$ is obtained by unrolling the dynamics:
$$
x_{k+1} = f(x_k, u_k), \qquad k = 0, \dots, N-1.
$$
The whole trajectory is therefore a function of $(x_0, U)$ only: we write $X = X(x_0, U)$.

**Question 2.** Implement `rollout(x0, U)` that takes $x_0 \in \mathbb{R}^{n_x}$ and $U \in \mathbb{R}^{N \times n_u}$ and returns $X \in \mathbb{R}^{(N+1) \times n_x}$.

In [ ]:
def rollout(x0, U):
    '''Unroll the dynamics from x0 with the controls U.

    Args:
        x0 (np.ndarray): initial state, shape (nx,).
        U (np.ndarray): controls, shape (N, nu).

    Returns:
        np.ndarray: states, shape (N+1, nx), with X[0] = x0.
    '''
    X = [x0]
    # ------------------------- TODO STUDENT (begin) -------------------------
    # TODO: unroll the dynamics, appending each new state to X
    # -------------------------- TODO STUDENT (end) --------------------------
    return np.array(X)

Let us apply a sinusoidal force on the cart, starting from the hanging position:

In [ ]:
N = 80
x0_down = np.array([0.0, np.pi, 0.0, 0.0])
U_sin = 5 * np.sin(2 * np.pi * np.arange(N) * dt)[:, None]
X_sin = rollout(x0_down, U_sin)

plot_trajectory(X_sin, dt, U_sin, title="Sinusoidal force")
play_trajectory(viz, X_sin, dt)

## 3. The optimal control problem

### 3.1 Formulation

We want to bring the cart-pole to the target state $x_{\rm target} = 0$ (cart at the origin, pole up, at rest) while using as little force as possible. We consider the discrete-time **Optimal Control Problem (OCP)** on a horizon of $N$ steps:
$$
\begin{aligned}
\min_{x_0, \dots, x_N,\; u_0, \dots, u_{N-1}} \quad & \sum_{k=0}^{N-1} \ell(x_k, u_k) + \ell_f(x_N) \\
\text{s.t.} \quad & x_{k+1} = f(x_k, u_k), \quad k = 0, \dots, N-1, \\
& x_0 = x_{\rm init} \ \text{(given)},
\end{aligned}
$$
with the quadratic **running cost** and **terminal cost**
$$
\ell(x, u) = \frac{\Delta t}{2}\Big( (x - x_{\rm target})^\top W\, (x - x_{\rm target}) + u^\top R\, u \Big), \qquad
\ell_f(x) = \frac{1}{2} (x - x_{\rm target})^\top W_f\, (x - x_{\rm target}),
$$
where $W, W_f \succeq 0$ and $R \succ 0$ are diagonal weight matrices. The factor $\Delta t$ makes the running cost a Riemann sum of the continuous cost $\int \frac12 (\|x - x_{\rm target}\|_W^2 + \|u\|_R^2)\,\mathrm{d}t$, so that the weights do not depend on the discretization.

### 3.2 Single shooting

Since $x_{0} = x_{\rm init}$ is given and $x_{k+1} = f(x_k, u_k)$, the states are *functions of the controls* only: $X(U) = (x_0(U), \dots, x_N(U))$. The constraints can be substituted in the cost, and the optimization problem is:
$$
\min_{U \in \mathbb{R}^{N n_u}} \; J(U) := \sum_{k=0}^{N-1} \ell\big(x_k(U), u_k\big) + \ell_f\big(x_N(U)\big).
$$
This is an **unconstrained** (nonlinear, nonconvex) optimization problem in dimension $N n_u$, that any generic solver can handle. This approach is called **single shooting**.

We gather the weights and the target in a small class. Its derivatives will be needed later (Section 5): since the cost is quadratic, they are straightforward.

**Question 3.** Complete the method `running_cost` of the class `QuadraticCost` (the other methods are given).

In [ ]:
class QuadraticCost:
    '''l(x, u) = dt/2 (|x - x*|_W^2 + |u|_R^2)   and   lf(x) = 1/2 |x - x*|_Wf^2.'''

    def __init__(self, W, R, Wf, x_target=None):
        self.W, self.R, self.Wf = W, R, Wf
        self.x_target = np.zeros(nx) if x_target is None else np.array(x_target)

    def running_cost(self, x, u):
        '''Running cost l(x, u) (a float).'''
        # ------------------------- TODO STUDENT (begin) -------------------------
        # TODO: running cost (a float)
        return None
        # -------------------------- TODO STUDENT (end) --------------------------

    def terminal_cost(self, x):
        '''Terminal cost lf(x) (a float).'''
        dx = x - self.x_target
        return 0.5 * dx @ self.Wf @ dx

    def total_cost(self, X, U):
        '''Total cost (running costs + terminal cost) of a trajectory X (N+1, nx) with controls U (N, nu).'''
        return float(np.sum([self.running_cost(x, u) for x, u in zip(X[:-1], U)]) + self.terminal_cost(X[-1]))

    # --- Derivatives (used from Section 5 on) ---
    def running_cost_derivatives(self, x, u):
        '''Return l_x, l_u, l_xx, l_uu, l_ux.'''
        dx = x - self.x_target
        return (dt * self.W @ dx, dt * self.R @ u,
                dt * self.W, dt * self.R, np.zeros((nu, nx)))

    def terminal_cost_derivatives(self, x):
        '''Return lf_x, lf_xx.'''
        return self.Wf @ (x - self.x_target), self.Wf

    def total_cost_derivatives(self, X, U):
        '''Partial gradients of the total cost, as if X and U were independent (dynamics ignored).

        Returns:
            L_X (N+1, nx): L_X[k] = l_x(x_k, u_k) for k < N, and L_X[N] = lf_x(x_N).
            L_U (N, nu): L_U[k] = l_u(x_k, u_k).
        '''
        running = [self.running_cost_derivatives(x, u) for x, u in zip(X[:-1], U)]
        L_X = np.array([l[0] for l in running] + [self.terminal_cost_derivatives(X[-1])[0]])
        L_U = np.array([l[1] for l in running])
        return L_X, L_U

In [ ]:
cost_stab = QuadraticCost(W=np.diag([1.0, 1.0, 0.1, 0.1]),
                          R=np.diag([1e-2]),
                          Wf=np.diag([100.0, 100.0, 10.0, 10.0]))

### 3.3 A first problem: stabilization

We start with an easy problem: the pole starts tilted by $0.15$ rad from the vertical, and we want to bring it back up within $1$ s. If your cost is correct, the cost of doing nothing ($U = 0$, the pole falls) is about $1240.5$.

In [ ]:
N_stab = 20  # 1 second
x0_stab = np.array([0.0, 0.15, 0.0, 0.0])

U_zero = np.zeros((N_stab, nu))
print("Cost of doing nothing:", cost_stab.total_cost(rollout(x0_stab, U_zero), U_zero))

## 4. Solving with a generic solver (SciPy) and finite differences

SciPy's [`minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html) works on a function of a *flat* vector $U \in \mathbb{R}^{N n_u}$. We use L-BFGS-B, a *quasi-Newton* method: it builds an approximation of the Hessian of $J$ from the successive gradients, and uses it to compute Newton-like steps. When no gradient is provided, SciPy approximates it by **finite differences**:
$$
\frac{\partial J}{\partial U_i}(U) \approx \frac{J(U + \varepsilon e_i) - J(U)}{\varepsilon}, \qquad i = 1, \dots, N n_u .
$$
Each gradient thus costs $N n_u + 1$ evaluations of $J$, *i.e.* $N n_u + 1$ rollouts of $N$ steps each: $\mathcal{O}(N^2)$ calls to the dynamics.

**Question 4.** Write the function `J(U_flat)` (the call to `minimize`, without gradient and starting from $U = 0$, is given).

In [ ]:
def J(U_flat, x0=x0_stab, cost=cost_stab):
    '''Cost J(U) of the controls U_flat (flat vector of size N * nu), from x0.'''
    U = U_flat.reshape(-1, nu)
    # ------------------------- TODO STUDENT (begin) -------------------------
    # TODO: total cost of the trajectory obtained from x0 with the controls U
    return None
    # -------------------------- TODO STUDENT (end) --------------------------


t0 = time.time()
res_fd = minimize(J, np.zeros(N_stab * nu), method="L-BFGS-B")
time_fd = time.time() - t0

print(res_fd.message)
print(f"cost = {res_fd.fun:.4f}   iterations = {res_fd.nit}   evaluations of J = {res_fd.nfev}   time = {time_fd:.2f} s")

In [ ]:
U_fd = res_fd.x.reshape(-1, nu)
X_fd = rollout(x0_stab, U_fd)
plot_trajectory(X_fd, dt, U_fd, title="Stabilization: L-BFGS + finite differences")
play_trajectory(viz, X_fd, dt)

It works, but look at the number of evaluations of $J$: each gradient costs $N n_u + 1 = 21$ evaluations, so almost all of them are finite-difference perturbations. Moreover, finite differences are *approximate*: the step $\varepsilon$ trades truncation error ($\mathcal{O}(\varepsilon)$) against round-off error ($\mathcal{O}(\text{machine precision}/\varepsilon)$, with a machine precision $\approx 10^{-16}$), which limits the precision the solver can reach.

## 5. Exact gradients with Pinocchio derivatives

### 5.1 Derivatives of the dynamics

Let us compute the Jacobians of the discrete dynamics
$$
A = \frac{\partial f}{\partial x}(x, u) \in \mathbb{R}^{n_x \times n_x}, \qquad
B = \frac{\partial f}{\partial u}(x, u) \in \mathbb{R}^{n_x \times n_u}.
$$
Pinocchio provides the analytical derivatives of the forward dynamics: `pin.computeABADerivatives(model, data, q, v, tau)` fills
$$
\texttt{data.ddq\_dq} = \frac{\partial a}{\partial q}, \qquad
\texttt{data.ddq\_dv} = \frac{\partial a}{\partial v}, \qquad
\texttt{data.Minv} = \frac{\partial a}{\partial \tau} = M(q)^{-1}.
$$
We now apply the chain rule to the semi-implicit Euler scheme. Since $\tau = S u$, we get $\frac{\partial a}{\partial u} = M^{-1} S$. Then, from $v^+ = v + \Delta t\, a$:
$$
\frac{\partial v^+}{\partial q} = \Delta t \frac{\partial a}{\partial q}, \qquad
\frac{\partial v^+}{\partial v} = I + \Delta t \frac{\partial a}{\partial v}, \qquad
\frac{\partial v^+}{\partial u} = \Delta t\, M^{-1} S,
$$
and from $q^+ = q + \Delta t\, v^+$ (do not forget that $v^+$ depends on $q$, $v$ and $u$!):
$$
\frac{\partial q^+}{\partial q} = I + \Delta t \frac{\partial v^+}{\partial q}, \qquad
\frac{\partial q^+}{\partial v} = \Delta t \frac{\partial v^+}{\partial v}, \qquad
\frac{\partial q^+}{\partial u} = \Delta t \frac{\partial v^+}{\partial u}.
$$
Finally, stacking the blocks:
$$
A = \begin{bmatrix} \frac{\partial q^+}{\partial q} & \frac{\partial q^+}{\partial v} \\[2pt] \frac{\partial v^+}{\partial q} & \frac{\partial v^+}{\partial v} \end{bmatrix}, \qquad
B = \begin{bmatrix} \frac{\partial q^+}{\partial u} \\[2pt] \frac{\partial v^+}{\partial u} \end{bmatrix}.
$$

*Remark.* For a robot whose configuration space is not Euclidean (free-flyer, continuous joints...), $q \oplus \Delta t v^+$ is not a sum anymore and its derivatives are given by `pin.dIntegrate`. Our cart-pole avoids this subtlety.

**Question 5.** Implement `step_derivatives(x, u)` returning $(A, B)$, of shapes $(n_x, n_x)$ and $(n_x, n_u)$. Hint: `np.block` and `np.vstack` help to assemble the blocks. Sanity check: expanding the formulas, $\frac{\partial q^+}{\partial q} = I + \Delta t^2 \frac{\partial a}{\partial q}$.

In [ ]:
def step_derivatives(x, u):
    '''Jacobians A = df/dx (nx, nx) and B = df/du (nx, nu) of the discrete dynamics.'''
    q, v = x[:nq], x[nq:]
    pin.computeABADerivatives(model, data, q, v, S @ u)
    da_dq, da_dv, da_du = data.ddq_dq, data.ddq_dv, data.Minv @ S
    A, B = None, None
    # ------------------------- TODO STUDENT (begin) -------------------------
    # TODO: derivatives of v+ = v + dt * a with respect to q, v and u
    # TODO: derivatives of q+ = q + dt * v+ (careful: v+ depends on q, v and u!), and assembly of A and B
    # -------------------------- TODO STUDENT (end) --------------------------
    return A, B

Always check your derivatives against finite differences (here, centered finite differences, whose error is $\mathcal{O}(\varepsilon^2)$):

In [ ]:
x_test, u_test = rng.standard_normal(nx), rng.standard_normal(nu)
A, B = step_derivatives(x_test, u_test)

eps = 1e-6
A_fd = np.array([(step(x_test + eps * e, u_test) - step(x_test - eps * e, u_test)) / (2 * eps)
                 for e in np.eye(nx)]).T
B_fd = np.array([(step(x_test, u_test + eps * e) - step(x_test, u_test - eps * e)) / (2 * eps)
                 for e in np.eye(nu)]).T
print("error on A:", np.abs(A - A_fd).max())
print("error on B:", np.abs(B - B_fd).max())  # both should be ~1e-10

### 5.2 Gradient of the cost: the adjoint method

The rollout is a chain of simple computations. Let us draw it as a graph:

<img src="tp4/adjoint_graph.png" alt="Computational graph of the rollout" width="850"/>

The control $u_k$ acts on the cost $J$ in two ways: **directly**, through the running cost $\ell_k = \ell(x_k, u_k)$, and **indirectly**, through the next state $x_{k+1} = f(x_k, u_k)$, which changes all the future costs. To account for the indirect effects, we introduce the sensitivity of the *future* cost to the state $x_k$:
$$
\lambda_k := \nabla_{x_k} J_k, \qquad \text{where} \quad J_k := \ell_k + \ell_{k+1} + \dots + \ell_{N-1} + \ell_f(x_N)
$$
is the **tail cost** (cost-to-go) from step $k$, the future states being given by the rollout. $\lambda_k$ is called the **adjoint** (or co-state). Reading the graph from right to left, the chain rule gives:

**Proposition (adjoint method).** With $A_k = \frac{\partial f}{\partial x}(x_k, u_k)$ and $B_k = \frac{\partial f}{\partial u}(x_k, u_k)$, for $k = N-1, \dots, 0$:
$$
\lambda_N = \nabla_x \ell_f(x_N), \qquad
\lambda_k = \underbrace{\nabla_x \ell_k}_{\text{direct}} + \underbrace{A_k^\top \lambda_{k+1}}_{\text{through } x_{k+1}}, \qquad
\nabla_{u_k} J = \underbrace{\nabla_u \ell_k}_{\text{direct}} + \underbrace{B_k^\top \lambda_{k+1}}_{\text{through } x_{k+1}} .
$$

In words: *the sensitivity of a node = its direct effect on the cost + what flows back from the next state, through the transposed Jacobian.* The direct terms are the partial derivatives of the cost, given by `cost.total_cost_derivatives(X, U)`.

<details>
<summary><b>Proof</b> (click to expand)</summary>

The tail cost satisfies $J_N = \ell_f(x_N)$ and $J_k = \ell(x_k, u_k) + J_{k+1}\big(f(x_k, u_k), u_{k+1}, \dots\big)$. Differentiating with respect to $x_k$ (chain rule) gives $\lambda_k = \nabla_x \ell_k + A_k^\top \lambda_{k+1}$. With respect to $u_k$: the terms $\ell_0, \dots, \ell_{k-1}$ of $J = J_0$ do not depend on $u_k$, and $u_k$ does not change $x_0, \dots, x_k$, hence $\nabla_{u_k} J = \nabla_{u_k} J_k = \nabla_u \ell_k + B_k^\top \lambda_{k+1}$. $\square$

</details>

<details>
<summary><b>For the curious:</b> the adjoints are Lagrange multipliers (click to expand)</summary>

Go back to the constrained problem of Section 3.1 and introduce a Lagrange multiplier $\lambda_{k+1}$ for each dynamics constraint: $\mathcal{L} = \sum_{k=0}^{N-1} \big[\ell(x_k, u_k) + \lambda_{k+1}^\top (f(x_k, u_k) - x_{k+1})\big] + \ell_f(x_N)$. The conditions $\nabla_{x_k} \mathcal{L} = 0$ ($k \ge 1$) are exactly the adjoint recursion, and at such a point $\nabla_{u_k} \mathcal{L} = \nabla_u \ell_k + B_k^\top \lambda_{k+1} = \nabla_{u_k} J$. The adjoints are the **Lagrange multipliers of the dynamics**.

</details>

This is exactly **back-propagation** (reverse-mode automatic differentiation) on the graph above. The gradient costs **one forward rollout and one backward pass**, *i.e.* $\mathcal{O}(N)$ instead of $\mathcal{O}(N^2)$ for finite differences.

**Question 6.** Implement the backward recursion in `J_and_grad(U_flat)` (the rollout and the direct derivatives are given). Careful: $\lambda_{k+1}$ must be used *before* being overwritten by $\lambda_k$.

In [ ]:
def J_and_grad(U_flat, x0=x0_stab, cost=cost_stab):
    '''Return J(U) and its gradient, computed with the adjoint method.'''
    U = U_flat.reshape(-1, nu)
    N = len(U)
    X = rollout(x0, U)
    L_X, L_U = cost.total_cost_derivatives(X, U)  # direct (partial) derivatives of the cost
    grad = np.zeros((N, nu))
    lam = L_X[N]
    for k in reversed(range(N)):
        A, B = step_derivatives(X[k], U[k])
        # ------------------------- TODO STUDENT (begin) -------------------------
        # TODO: gradient with respect to u_k (it uses lambda_{k+1})
        # TODO: adjoint lambda_k
        # -------------------------- TODO STUDENT (end) --------------------------
    return cost.total_cost(X, U), grad.ravel()

In [ ]:
U_test = 0.1 * rng.standard_normal(N_stab * nu)
_, g = J_and_grad(U_test)
g_fd = approx_fprime(U_test, J, 1e-7)
print("relative error on the gradient:", norm(g - g_fd) / norm(g))  # should be ~1e-7 (limited by finite differences)

Let us compare the cost of one gradient computation for several horizons $N$:

In [ ]:
for N_test in [20, 40, 80, 160]:
    U_test = 0.1 * rng.standard_normal(N_test * nu)
    t0 = time.time(); approx_fprime(U_test, J, 1e-7); t_fd = time.time() - t0
    t0 = time.time(); J_and_grad(U_test); t_adj = time.time() - t0
    print(f"N = {N_test:4d}:   finite differences {1e3 * t_fd:7.1f} ms    adjoint {1e3 * t_adj:6.1f} ms")

Now we give the exact gradient to the solver (`jac=True` tells SciPy that the function returns both the value and the gradient):

In [ ]:
t0 = time.time()
res_adj = minimize(J_and_grad, np.zeros(N_stab * nu), jac=True, method="L-BFGS-B")
time_adj = time.time() - t0

print(res_adj.message)
print(f"cost = {res_adj.fun:.4f}   iterations = {res_adj.nit}   evaluations = {res_adj.nfev}   time = {time_adj:.2f} s")

In [ ]:
U_adj = res_adj.x.reshape(-1, nu)
X_adj = rollout(x0_stab, U_adj)
plot_trajectory(X_adj, dt, U_adj, title="Stabilization: L-BFGS + exact gradient")
play_trajectory(viz, X_adj, dt)

### 5.3 A harder problem: the swing-up

Let us now start from the pole hanging down, $x_{\rm init} = (0, \pi, 0, 0)$, and ask the pole to be up after $3$ s. We lower the running weights $W$: we do not care much about the path, we mostly want to *arrive* at the target (encoded by the large terminal weight $W_f$). This takes a few seconds (the exact numbers may vary slightly from one machine to another).

In [ ]:
N_swing = 60  # 3 seconds
x0_swing = np.array([0.0, np.pi, 0.0, 0.0])
cost_swing = QuadraticCost(W=np.diag([0.1, 0.1, 0.01, 0.01]),
                           R=np.diag([1e-2]),
                           Wf=np.diag([100.0, 100.0, 10.0, 10.0]))

t0 = time.time()
# The line search of L-BFGS tries very large steps, for which the pole diverges (overflow): we silence these warnings.
with np.errstate(over="ignore", invalid="ignore"):
    res_swing = minimize(J_and_grad, np.zeros(N_swing * nu), args=(x0_swing, cost_swing),
                         jac=True, method="L-BFGS-B", options=dict(maxiter=10000))
time_swing_lbfgs = time.time() - t0
print(res_swing.message)
print(f"cost = {res_swing.fun:.4f}   iterations = {res_swing.nit}   time = {time_swing_lbfgs:.2f} s")

U_swing_lbfgs = res_swing.x.reshape(-1, nu)
X_swing_lbfgs = rollout(x0_swing, U_swing_lbfgs)
print("final state:", X_swing_lbfgs[-1])
plot_trajectory(X_swing_lbfgs, dt, U_swing_lbfgs, title="Swing-up: L-BFGS + exact gradient")
play_trajectory(viz, X_swing_lbfgs, dt)

The generic solver needs thousands of iterations. **Why?** Think about it before reading further.

The reason is that the problem is badly conditioned: a change of $u_0$ has a huge effect on $x_N$ (the system is unstable), while a change of $u_{N-1}$ has almost none. A quasi-Newton method, which ignores the structure of the problem, has a hard time building a good model of the Hessian. Newton's method would fix this, but the Hessian $\nabla^2 J$ is a dense $(N n_u) \times (N n_u)$ matrix: solving the Newton system costs $\mathcal{O}(N^3)$. Let us now **exploit the temporal structure** of the OCP to get a (Gauss-)Newton-like step for only $\mathcal{O}(N)$.

## 6. iLQR: iterative Linear Quadratic Regulator

### 6.1 Bellman's principle and the value function

Define the **value function** (optimal cost-to-go) at time $k$:
$$
V_k(x) = \min_{u_k, \dots, u_{N-1}} J_k(x, u_k, \dots, u_{N-1}).
$$

**Proposition (Bellman's principle).** $V_N = \ell_f$ and, for $k = N-1, \dots, 0$,
$$
V_k(x) = \min_{u} \; \underbrace{\ell(x, u) + V_{k+1}\big(f(x, u)\big)}_{=: \; \mathcal{Q}_k(x, u)} .
$$

<details>
<summary><b>Proof</b> (click to expand)</summary>

With the recursion $J_k = \ell(x, u_k) + J_{k+1}(f(x, u_k), u_{k+1}, \dots)$ of Section 5.2, split the minimization over $u_k$ and over the tail $(u_{k+1}, \dots)$. Since $\ell(x, u_k)$ does not depend on the tail,
$V_k(x) = \min_{u_k} \big[ \ell(x, u_k) + \min_{u_{k+1}, \dots} J_{k+1}(f(x, u_k), u_{k+1}, \dots) \big] = \min_{u_k} \big[ \ell(x, u_k) + V_{k+1}(f(x, u_k)) \big]$. $\square$

</details>

The function $\mathcal{Q}_k$ is called the **Q-function** (action-value function); its derivatives will be written $Q_x, Q_u, Q_{xx}, \dots$ (this is why the state weight, often called $Q$ in LQR, is written $W$ here). In general, $V_k$ cannot be computed exactly: it is a function on the whole state space! The idea of iLQR is to compute it only **locally, around a nominal trajectory**, with a second-order model.

**Why a quadratic model?** If $V_{k+1}$ is quadratic, $f$ is linear and $\ell$ is quadratic, then $\mathcal{Q}_k$ is quadratic in $(x, u)$, and its minimum over $u$ is again a quadratic function of $x$ (provided $Q_{uu} \succ 0$, see below): $V_k$ is quadratic. By induction, for linear dynamics and quadratic costs (the *LQ problem*), all the value functions are exactly quadratic and Bellman's recursion can be computed exactly: this is the classical **LQR**. For our nonlinear system, iLQR (i) builds the LQ approximation of the OCP around the nominal trajectory, (ii) solves it exactly with Bellman's recursion (**backward pass**), (iii) applies the resulting controller on the nonlinear system to get a new trajectory (**forward pass**), and repeats:

<img src="tp4/ilqr_scheme.png" alt="iLQR scheme" width="850"/>

In Section 5.2, the adjoint $\lambda_k = \nabla_{x_k} J_k$ carried *first-order* information backward in time. The backward pass of iLQR carries a *second-order* model: a gradient $V_x$ **and** a Hessian $V_{xx}$.

### 6.2 The local LQ problem

Let $(\bar X, \bar U)$ be the current nominal trajectory (obtained by a rollout). We look for a correction $\delta u_k$ as a function of the deviation $\delta x_k = x_k - \bar x_k$. We compute the value function of the **LQ approximation** of the OCP around $(\bar X, \bar U)$, in the variables $(\delta x, \delta u)$. Assume that, at step $k+1$, it is the quadratic function
$$
V_{k+1}(\bar x_{k+1} + \delta x) \approx V_{k+1}(\bar x_{k+1}) + V'_x{}^\top \delta x + \frac12 \delta x^\top V'_{xx} \delta x ,
$$
where the primes denote the gradient $V'_x$ and Hessian $V'_{xx}$ of the model at step $k+1$.

**Taylor expansions.** Around $(\bar x_k, \bar u_k)$, we expand the running cost to second order,
$$
\ell(\bar x_k + \delta x, \bar u_k + \delta u) \approx \ell(\bar x_k, \bar u_k) + \ell_x^\top \delta x + \ell_u^\top \delta u
+ \frac12 \delta x^\top \ell_{xx} \delta x + \delta u^\top \ell_{ux} \delta x + \frac12 \delta u^\top \ell_{uu} \delta u ,
$$
and the dynamics to **first order** only:
$$
f(\bar x_k + \delta x, \bar u_k + \delta u) \approx \bar x_{k+1} + \underbrace{A_k \delta x + B_k \delta u}_{=: \; \delta x'} .
$$
Plugging $\delta x' = A_k \delta x + B_k \delta u$ in the model of $V_{k+1}$ gives
$$
V_{k+1}(\bar x_{k+1} + \delta x') \approx \text{cst} + V'_x{}^\top (A_k \delta x + B_k \delta u) + \frac12 (A_k \delta x + B_k \delta u)^\top V'_{xx} (A_k \delta x + B_k \delta u).
$$
Summing the two expansions and collecting the terms in $\delta x$, $\delta u$, $\delta x\,\delta x$, $\delta u\,\delta u$ and $\delta u\,\delta x$, the Q-function becomes a quadratic function of $(\delta x, \delta u)$:
$$
\mathcal{Q}_k(\bar x_k + \delta x, \bar u_k + \delta u) \approx \text{cst} +
\begin{bmatrix} Q_x \\ Q_u \end{bmatrix}^\top \begin{bmatrix} \delta x \\ \delta u \end{bmatrix}
+ \frac12 \begin{bmatrix} \delta x \\ \delta u \end{bmatrix}^\top
\begin{bmatrix} Q_{xx} & Q_{ux}^\top \\ Q_{ux} & Q_{uu} \end{bmatrix}
\begin{bmatrix} \delta x \\ \delta u \end{bmatrix},
$$
with
$$
\boxed{
\begin{aligned}
Q_x &= \ell_x + A_k^\top V'_x, & Q_{xx} &= \ell_{xx} + A_k^\top V'_{xx} A_k, \\
Q_u &= \ell_u + B_k^\top V'_x, & Q_{uu} &= \ell_{uu} + B_k^\top V'_{xx} B_k, \\
& & Q_{ux} &= \ell_{ux} + B_k^\top V'_{xx} A_k,
\end{aligned}}
$$
where all derivatives of $\ell$ are evaluated at $(\bar x_k, \bar u_k)$. Expanding $f$ to first order only is the *Gauss-Newton* approximation (see "To go further"). Note that if we drop all the second-order terms ($V_{xx} \equiv 0$ at every step), then (since $\ell_{ux} = 0$ for our cost) $Q_{ux} = 0$ and $Q_x = \ell_x + A_k^\top V'_x$, $Q_u = \ell_u + B_k^\top V'_x$: exactly the adjoint recursion of Section 5.2.

### 6.3 Minimization and update of the value function

**Minimization over $\delta u$.** If $Q_{uu} \succ 0$, the quadratic function is strictly convex in $\delta u$ and its minimizer is given by $\nabla_{\delta u} \mathcal{Q}_k = 0$, *i.e.* $Q_u + Q_{ux}\delta x + Q_{uu} \delta u = 0$:
$$
\boxed{\delta u^\star(\delta x) = \kappa_k + K_k\, \delta x, \qquad \kappa_k = -Q_{uu}^{-1} Q_u, \qquad K_k = -Q_{uu}^{-1} Q_{ux}.}
$$
$\kappa_k \in \mathbb{R}^{n_u}$ is a **feedforward** term and $K_k \in \mathbb{R}^{n_u \times n_x}$ a **feedback gain**.

**Update of the value function.** By Bellman, $V_k(\bar x_k + \delta x) \approx \mathcal{Q}_k(\bar x_k + \delta x, \bar u_k + \kappa_k + K_k \delta x)$. Plugging $\delta u = \kappa_k + K_k \delta x$ in the quadratic model and collecting the terms in $\delta x$ and $\delta x\,\delta x$:
$$
\begin{aligned}
V_x &= Q_x + K_k^\top Q_u + Q_{ux}^\top \kappa_k + K_k^\top Q_{uu} \kappa_k, \\
V_{xx} &= Q_{xx} + K_k^\top Q_{ux} + Q_{ux}^\top K_k + K_k^\top Q_{uu} K_k .
\end{aligned}
$$
Using $Q_{uu} \kappa_k = -Q_u$ and $Q_{uu} K_k = -Q_{ux}$ (check it with pen and paper!), this simplifies to
$$
\boxed{V_x = Q_x + Q_{ux}^\top \kappa_k, \qquad V_{xx} = Q_{xx} - K_k^\top Q_{uu} K_k .}
$$
The recursion is initialized with $V_x = \nabla_x \ell_f(\bar x_N)$, $V_{xx} = \nabla^2_{xx} \ell_f(\bar x_N)$ and run backward from $k = N-1$ down to $0$.

<details>
<summary><b>For the curious:</b> why we must symmetrize $V_{xx}$ in the code (click to expand)</summary>

In floating-point arithmetic, products such as $A_k^\top V'_{xx} A_k$ are not exactly symmetric. A quadratic form $\delta x^\top V \delta x$ does not "see" the antisymmetric part of $V$, so the Riccati correction never damps it: it is simply propagated as $A_k^\top (\cdot) A_k$, and amplified exponentially since $A_k$ is unstable around the upright position. After a few hundred steps, it spoils the gains completely. The cure is cheap: after each update, $V_{xx} \leftarrow \frac12 (V_{xx} + V_{xx}^\top)$.

</details>

**Predicted decrease.** Applying $\delta u_k = \alpha \kappa_k + K_k \delta x_k$ to the LQ model changes the cost by $\sum_k \big(\alpha\, Q_{u,k}^\top \kappa_k + \frac{\alpha^2}{2} \kappa_k^\top Q_{uu,k} \kappa_k\big) = (\alpha - \frac{\alpha^2}{2})\, d$, where
$$
d := \sum_{k=0}^{N-1} Q_{u,k}^\top \kappa_k = -\sum_{k=0}^{N-1} Q_{u,k}^\top Q_{uu,k}^{-1} Q_{u,k} \;\le\; 0 .
$$

**Is $Q_{uu}$ invertible?** Yes, always, for our cost: $Q_{uu} \succeq \Delta t R \succ 0$, so the backward pass is well defined.

<details>
<summary><b>For the curious:</b> proof that $Q_{uu} \succeq \Delta t R$ (click to expand)</summary>

We show by backward induction that $V_{xx} \succeq 0$ for all $k$ (assuming $W, W_f \succeq 0$, $R \succ 0$). At $k = N$, $V_{xx} = W_f \succeq 0$. Assume $V'_{xx} \succeq 0$. Since $\ell_{ux} = 0$, the Hessian of the Q-function is $H = \mathrm{blockdiag}(\Delta t W, \Delta t R) + [A_k\ B_k]^\top V'_{xx} [A_k\ B_k] \succeq 0$, so in particular $Q_{uu} \succeq \Delta t R \succ 0$. Moreover, by construction of $\kappa_k, K_k$, $\frac12 \delta x^\top V_{xx} \delta x = \min_{\delta u} \frac12 [\delta x; \delta u]^\top H [\delta x; \delta u] \ge 0$ for all $\delta x$, hence $V_{xx} \succeq 0$. $\square$

</details>

**Question 7.** Implement the backward pass. For $k$ from $N-1$ down to $0$:
1. compute $A_k, B_k$ (`step_derivatives`) and the derivatives of $\ell$ (`cost.running_cost_derivatives`);
2. compute $Q_x, Q_u, Q_{xx}, Q_{uu}, Q_{ux}$ (with the *old* $V_x, V_{xx}$);
3. compute $\kappa_k$ and $K_k$ (use `np.linalg.solve` rather than an explicit inverse) and store them in `kappas[k]`, `Ks[k]`;
4. update $V_x$ and $V_{xx}$, then **symmetrize** $V_{xx} \leftarrow \frac12 (V_{xx} + V_{xx}^\top)$ (see the box above);
5. accumulate $d$.

Shapes (with $n_u = 1$, watch out for broadcasting!): $Q_u$ and $\kappa_k$ have shape `(nu,)`, $Q_{uu}$ `(nu, nu)`, $Q_{ux}$ and $K_k$ `(nu, nx)`.

In [ ]:
def backward_pass(X, U, cost):
    '''iLQR backward pass around the nominal trajectory (X, U).

    Returns:
        kappas (N, nu): feedforward terms, Ks (N, nu, nx): feedback gains,
        d (float): sum_k Q_u^T kappa_k (<= 0), the predicted decrease is (alpha - alpha^2 / 2) d.
    '''
    N = len(U)
    kappas, Ks = np.zeros((N, nu)), np.zeros((N, nu, nx))
    d = 0.0
    V_x, V_xx = cost.terminal_cost_derivatives(X[-1])
    for k in reversed(range(N)):
        A, B = step_derivatives(X[k], U[k])
        l_x, l_u, l_xx, l_uu, l_ux = cost.running_cost_derivatives(X[k], U[k])

        # ------------------------- TODO STUDENT (begin) -------------------------
        # TODO: derivatives of the Q-function at step k
        # TODO: feedforward term and feedback gain, stored in kappas[k] and Ks[k]
        # TODO: update of the value function V_x, V_xx
        V_xx = 0.5 * (V_xx + V_xx.T)  # remove the antisymmetric round-off errors
        # TODO: accumulation of d
        # -------------------------- TODO STUDENT (end) --------------------------

        assert V_x.shape == (nx,) and V_xx.shape == (nx, nx), "wrong shapes: check the broadcasting of your formulas"
    return kappas, Ks, float(d)

*Remark (Riccati).* In the LQ case ($A_k = A$, $B_k = B$, $P_k := V_{xx}$), the update of $V_{xx}$ reads
$$
P_k = \Delta t\, W + A^\top P_{k+1} A - A^\top P_{k+1} B \big(\Delta t\, R + B^\top P_{k+1} B\big)^{-1} B^\top P_{k+1} A ,
$$
the **discrete Riccati equation** of the LQR. iLQR solves a sequence of LQR problems around successive trajectories, hence its name. For a long horizon ($N - k \to \infty$), $P_k$ converges to the solution $P_\infty$ of the *algebraic* Riccati equation, and $K_k$ to the stationary LQR gain $K_\infty = -(\Delta t R + B^\top P_\infty B)^{-1} B^\top P_\infty A$.

Two sanity checks of your backward pass:
1. At the optimum found by L-BFGS in Section 5, $\nabla_U J \approx 0$, so $Q_u \approx 0$ at every step (backward induction: $Q_u = \nabla_{u_k} J$ and $V_x = \lambda_k$ when all $\kappa_k = 0$), hence $\kappa_k \approx 0$ and $d \approx 0$: expect $|\kappa_k| \ll 1$ N (the controls are of order $10$ N) and $|d| \lesssim 10^{-5}$ (the tolerance of L-BFGS).
2. Around the equilibrium (where $A_k = A$, $B_k = B$ at every step) and with a long horizon, $K_0$ should match $K_\infty$ computed with `scipy.linalg.solve_discrete_are`: expect an error $\sim 10^{-12}$.

In [ ]:
kappas, Ks, d = backward_pass(X_adj, U_adj, cost_stab)
print("At the L-BFGS optimum:   max |kappa_k| =", np.abs(kappas).max(), "   d =", d)

A_eq, B_eq = step_derivatives(np.zeros(nx), np.zeros(nu))  # linearization at the upright equilibrium
W_dt, R_dt = dt * cost_stab.W, dt * cost_stab.R
P_inf = solve_discrete_are(A_eq, B_eq, W_dt, R_dt)
K_lqr = -np.linalg.solve(R_dt + B_eq.T @ P_inf @ B_eq, B_eq.T @ P_inf @ A_eq)
_, Ks_eq, _ = backward_pass(np.zeros((201, nx)), np.zeros((200, nu)), cost_stab)
print("Error between K_0 and the LQR gain:", np.abs(Ks_eq[0] - K_lqr).max())

### 6.4 The forward pass and the line search

The backward pass gives a *local* policy $\delta u_k = \kappa_k + K_k \delta x_k$. The new trajectory is obtained by applying it on the **true nonlinear dynamics**:
$$
x^{\rm new}_0 = x_{\rm init}, \qquad
u^{\rm new}_k = \bar u_k + \alpha\, \kappa_k + K_k (x^{\rm new}_k - \bar x_k), \qquad
x^{\rm new}_{k+1} = f(x^{\rm new}_k, u^{\rm new}_k).
$$
Two remarks:
- The feedback term $K_k(x^{\rm new}_k - \bar x_k)$ keeps the new trajectory close to the nominal one, where the LQ model is valid. This is important for unstable systems such as ours: applying only the open-loop correction $\bar u_k + \alpha \kappa_k$ may make the trajectory diverge.
- The LQ model is only an approximation: a full step $\alpha = 1$ may increase the cost. We use a **backtracking line search**: start from $\alpha = 1$ and halve it until the cost decreases.

<details>
<summary><b>For the curious:</b> why the line search succeeds (click to expand)</summary>

With $\alpha = 0$, the forward pass returns exactly $(\bar X, \bar U)$. One can check that the derivative of the cost with respect to $\alpha$ at $\alpha = 0$ is $d < 0$ (the LQ model is exact to first order), so a small enough step always decreases the cost.

</details>

**Question 8.** Implement the forward pass.

In [ ]:
def forward_pass(x0, X, U, kappas, Ks, alpha):
    '''Roll out the policy u_k = U_k + alpha kappa_k + K_k (x_k - X_k) on the nonlinear dynamics.'''
    N = len(U)
    X_new, U_new = np.zeros_like(X), np.zeros_like(U)
    X_new[0] = x0
    # ------------------------- TODO STUDENT (begin) -------------------------
    for k in range(N):
        # TODO: control given by the policy at step k
        # TODO: next state on the nonlinear dynamics
        pass
    # -------------------------- TODO STUDENT (end) --------------------------
    return X_new, U_new

### 6.5 The iLQR algorithm

Putting everything together:

> **Algorithm (iLQR).** Given $x_{\rm init}$ and an initial guess $\bar U$:
> 1. Rollout: $\bar X = X(x_{\rm init}, \bar U)$.
> 2. Repeat until convergence:
>    1. Backward pass around $(\bar X, \bar U)$ $\to$ $(\kappa_k, K_k)_k$ and $d$.
>    2. If $|d| < \text{tol}$: stop.
>    3. Forward pass with line search on $\alpha$ $\to$ $(X^{\rm new}, U^{\rm new})$ with $J(U^{\rm new}) < J(\bar U)$.
>    4. $(\bar X, \bar U) \leftarrow (X^{\rm new}, U^{\rm new})$.

<details>
<summary><b>For the curious:</b> why $d = 0$ is a good stopping criterion (click to expand)</summary>

Since $Q_{uu} \succ 0$, $d = -\sum_k Q_u^\top Q_{uu}^{-1} Q_u = 0$ implies $Q_u = 0$, hence $\kappa_k = 0$ for all $k$. Then $V_x = Q_x = \ell_x + A_k^\top V'_x$ follows exactly the adjoint recursion, so $V_x = \lambda_k$ and $Q_u = \ell_u + B_k^\top \lambda_{k+1} = \nabla_{u_k} J = 0$: $\bar U$ is a **stationary point** of $J$.

</details>

The function is given:

In [ ]:
def ilqr(x0, U_init, cost, max_iter=200, tol=1e-6, verbose=False):
    '''Solve the OCP with iLQR from the initial guess U_init (N, nu).

    Returns:
        X (N+1, nx), U (N, nu): optimal trajectory,
        Ks (N, nu, nx): feedback gains, costs (list): cost at each iteration.
    '''
    U = U_init.copy()
    with np.errstate(over="ignore", invalid="ignore"):
        X = rollout(x0, U)
        costs = [cost.total_cost(X, U)]
    if not np.isfinite(costs[0]):
        print("Warning: the initial rollout diverges, iLQR cannot start.")
        return X, U, np.zeros((len(U), nu, nx)), costs
    for it in range(max_iter):
        kappas, Ks, d = backward_pass(X, U, cost)
        if abs(d) < tol:
            break
        alpha = 1.0
        # A full step may make the unstable pole diverge: the cost is then inf/nan and the step rejected.
        with np.errstate(over="ignore", invalid="ignore"):
            while alpha > 1e-6:
                X_new, U_new = forward_pass(x0, X, U, kappas, Ks, alpha)
                new_cost = cost.total_cost(X_new, U_new)
                if new_cost < costs[-1]:
                    break
                alpha /= 2
            else:
                if verbose:
                    print("Line search failed.")
                break
        X, U = X_new, U_new
        costs.append(new_cost)
        if verbose:
            print(f"iter {it:3d}   cost = {new_cost:.6f}   alpha = {alpha:.3f}   d = {d:.2e}")
    else:  # stopped on max_iter: recompute the gains around the returned trajectory
        _, Ks, _ = backward_pass(X, U, cost)
    return X, U, Ks, costs

### 6.6 Back to the stabilization problem

Let us solve the stabilization problem of Section 3.3 and compare with SciPy. You should get convergence in about 5 iterations, to the same cost as L-BFGS ($\approx 0.1642$).

In [ ]:
t0 = time.time()
X_ilqr, U_ilqr, Ks_ilqr, costs_ilqr = ilqr(x0_stab, np.zeros((N_stab, nu)), cost_stab, verbose=True)
time_ilqr = time.time() - t0

print(f"\nL-BFGS + finite differences: cost = {res_fd.fun:.4f}   time = {time_fd:.2f} s")
print(f"L-BFGS + exact gradient:     cost = {res_adj.fun:.4f}   time = {time_adj:.2f} s")
print(f"iLQR:                        cost = {costs_ilqr[-1]:.4f}   time = {time_ilqr:.2f} s   ({len(costs_ilqr) - 1} iterations)")

In [ ]:
plot_trajectory(X_ilqr, dt, U_ilqr, title="Stabilization: iLQR")
play_trajectory(viz, X_ilqr, dt)

The three methods find the same optimum, but iLQR needs only a handful of iterations, with full steps $\alpha = 1$: close to the solution, its convergence is very fast, like Newton's method. (iLQR is a Gauss-Newton method: its convergence is quadratic only if the neglected second-order terms vanish at the solution; otherwise it is fast but linear.)

#### Optional: local minima

The OCP is *not convex* (the dynamics is nonlinear), so all these methods only find **local minima**, which depend on the initial guess. Let us start from a larger tilt, $\theta_0 = 0.25$ rad. With $U = 0$, the initial rollout lets the pole fall down, and iLQR converges to a poor local minimum where the pole falls and swings back up. (The feedback term of the forward pass keeps iLQR close to this bad initial rollout.) A good initial guess (here the L-BFGS solution) solves the issue.

In [ ]:
x0_hard = np.array([0.0, 0.25, 0.0, 0.0])

X_bad, U_bad, _, costs_bad = ilqr(x0_hard, np.zeros((N_stab, nu)), cost_stab)
print(f"iLQR from U = 0:          cost = {costs_bad[-1]:.4f}   max |theta| = {np.abs(X_bad[:, 1]).max():.2f} rad")

res_hard = minimize(J_and_grad, np.zeros(N_stab * nu), args=(x0_hard, cost_stab), jac=True, method="L-BFGS-B")
X_good, U_good, _, costs_good = ilqr(x0_hard, res_hard.x.reshape(-1, nu), cost_stab)
print(f"iLQR from L-BFGS guess:   cost = {costs_good[-1]:.4f}   max |theta| = {np.abs(X_good[:, 1]).max():.2f} rad")

print("Bad local minimum:")
play_trajectory(viz, X_bad, dt)
print("Good initial guess:")
play_trajectory(viz, X_good, dt)

### 6.7 The swing-up with iLQR

Now the real test: the swing-up of Section 5.3, from $U = 0$. Compare the time and the number of iterations with L-BFGS.

In [ ]:
t0 = time.time()
X_swing, U_swing, Ks_swing, costs_swing = ilqr(x0_swing, np.zeros((N_swing, nu)), cost_swing)
time_swing_ilqr = time.time() - t0

print(f"L-BFGS + exact gradient: cost = {res_swing.fun:.4f}   time = {time_swing_lbfgs:.2f} s   ({res_swing.nit} iterations)")
print(f"iLQR:                    cost = {costs_swing[-1]:.4f}   time = {time_swing_ilqr:.2f} s   ({len(costs_swing) - 1} iterations)")
print("final state:", X_swing[-1])

plt.semilogy(np.array(costs_swing) - costs_swing[-1] + 1e-10)
plt.xlabel("iteration"); plt.ylabel(r"$J(U) - J^\star$"); plt.title("iLQR convergence (swing-up)"); plt.grid(True)
plt.show()
plot_trajectory(X_swing, dt, U_swing, title="Swing-up: iLQR")

In [ ]:
play_trajectory(viz, X_swing, dt)

Note that the solution is obtained from $U = 0$ as initial guess, without any hand-crafted "energy pumping" strategy: the swing-up motion *emerges* from the optimization.

Two observations. First, L-BFGS stopped at a worse point than iLQR (another local minimum, or stagnation of L-BFGS on this badly conditioned problem): another illustration of the nonconvexity. Second, iLQR needs many more iterations than for the stabilization: far from the solution, the LQ model is poor, the line search takes small steps $\alpha < 1$, and progress is slow (rerun `ilqr` with `verbose=True` to see the steps $\alpha$). The fast convergence only appears at the end, see the convergence plot.

To sum up the whole notebook in one picture:

In [ ]:
methods = ["L-BFGS\n+ finite diff.", "L-BFGS\n+ adjoint", "iLQR"]
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
bars = axs[0].bar(methods, [time_fd, time_adj, time_ilqr], color=["tab:gray", "tab:blue", "tab:orange"])
axs[0].bar_label(bars, fmt="%.3f s")
axs[0].set_yscale("log"); axs[0].set_ylabel("time (s)"); axs[0].set_title("Stabilization (N = 20)")
bars = axs[1].bar(methods[1:], [time_swing_lbfgs, time_swing_ilqr], color=["tab:blue", "tab:orange"])
axs[1].bar_label(bars, fmt="%.2f s")
axs[1].set_yscale("log"); axs[1].set_ylabel("time (s)"); axs[1].set_title("Swing-up (N = 60)")
for ax in axs:
    ax.grid(True, axis="y", which="both", alpha=0.3)
    ax.margins(y=0.25)
plt.tight_layout()
plt.show()

## 7. Closing the loop - BONUS

### 7.1 Open loop vs. feedback

The trajectory $(X^\star, U^\star)$ is computed *offline*. If we simply replay $U^\star$ on the robot (**open loop**), the smallest perturbation, model error or numerical error will make the unstable pole fall. Fortunately, iLQR gives us more than a trajectory: the gains $K_k$ define a **locally optimal feedback policy**
$$
u_k = u^\star_k + K_k\,(x_k - x^\star_k),
$$
which is the time-varying LQR controller of the linearization around the optimal trajectory.

We simulate the swing-up with a push on the pole at $t = 1.5$ s (a sudden change of $\dot\theta$), and continue the simulation $2$ s after the end of the horizon. After the horizon, we need a controller to hold the pole up. The last gain $K_{N-1}$ is a bad choice: it only "sees" one step ahead before the terminal cost. We use instead the **stationary LQR gain** $K_\infty$ around $x_{\rm target}$, obtained as in the sanity check of Section 6.3 by a backward pass on a long horizon around the equilibrium. Let us compare the eigenvalues of the closed-loop linearized dynamics $A + B K$ for both gains:

In [ ]:
A_eq, B_eq = step_derivatives(np.zeros(nx), np.zeros(nu))  # linearization at the upright equilibrium (as in Section 6.3)
_, Ks_hold, _ = backward_pass(np.zeros((201, nx)), np.zeros((200, nu)), cost_swing)
K_inf = Ks_hold[0]
print("|eig(A + B K_N-1)| =", np.abs(np.linalg.eigvals(A_eq + B_eq @ Ks_swing[-1])))
print("|eig(A + B K_inf)| =", np.abs(np.linalg.eigvals(A_eq + B_eq @ K_inf)))

With $K_{N-1}$, one eigenvalue has a modulus $> 1$: the pole would slowly fall. With $K_\infty$, all the eigenvalues have a modulus $< 1$: the upright equilibrium is (locally) asymptotically stable.

**Question 9.** Complete `simulate(closed_loop)` (the open-loop part is given: $u_k = u^\star_k$ during the horizon and $u = 0$ after). In closed loop, apply $u_k = u^\star_k + K_k (x_k - x^\star_k)$ during the horizon and $u = K_\infty (x - x^\star_N)$ after, where $x^\star_N$ = `X_ref[-1]` is the end of the optimal trajectory.

In [ ]:
N_after = 40  # 2 s after the end of the horizon
k_push = 30   # push at t = 1.5 s
push = np.array([0.0, 0.0, 0.0, 1.0])  # +1 rad/s on the pole velocity


def simulate(closed_loop, X_ref=X_swing, U_ref=U_swing, Ks=Ks_swing, K_inf=K_inf):
    x = X_ref[0].copy()
    X_sim = [x]
    for k in range(len(U_ref) + N_after):
        if k == k_push:
            x = x + push
        if k < len(U_ref):
            u = U_ref[k].copy()  # copy: do not modify U_ref in place!
            if closed_loop:
                # ------------------------- TODO STUDENT (begin) -------------------------
                # TODO: closed-loop control during the horizon
                u = None
                # -------------------------- TODO STUDENT (end) --------------------------
        else:
            u = np.zeros(nu)
            if closed_loop:
                # ------------------------- TODO STUDENT (begin) -------------------------
                # TODO: closed-loop control after the horizon
                u = None
                # -------------------------- TODO STUDENT (end) --------------------------
        x = step(x, u)
        X_sim.append(x)
    return np.array(X_sim)


X_open = simulate(closed_loop=False)
X_closed = simulate(closed_loop=True)

t = np.arange(len(X_open)) * dt
plt.plot(t, X_open[:, 1], label="open loop")
plt.plot(t, X_closed[:, 1], label="closed loop (iLQR gains)")
plt.axvline(k_push * dt, color="k", ls="--", label="push")
plt.xlabel("time (s)"); plt.ylabel(r"$\theta$ (rad)"); plt.legend(); plt.grid(True)
plt.show()

In [ ]:
print("Open loop")
play_trajectory(viz, X_open, dt)

In [ ]:
print("Closed loop")
play_trajectory(viz, X_closed, dt)

Try stronger pushes (`push = [0, 0, 0, 4]`): the feedback is only *locally* optimal, and fails when the state goes too far from the nominal trajectory.

### 7.2 Model Predictive Control (MPC)

To handle larger perturbations, we can **re-plan online**: at each control step, solve the OCP from the *current* measured state on a **receding horizon** of $H$ steps, apply only the first control, and repeat.

> **Algorithm (MPC).** At each time step $k$:
> 1. Measure the current state $x_k$.
> 2. Solve the OCP on the horizon $[k, k+H]$ starting from $x_k$, warm-started with the previous solution.
> 3. Apply the first control of the plan to the system.
> 4. Shift the solution by one step to prepare the next warm start.

Solving the OCP to convergence at each step would be too slow for a real robot. Thanks to the warm start, the previous solution is already almost optimal, and **a few iLQR iterations** per step are enough. The first problem is solved to convergence before starting.

**Warm start (given).** After a push, replaying the previous controls in open loop would let the pole fall and trap iLQR in a bad local minimum (remember Section 6.6). We therefore roll out the previous **feedback policy** (controls `U_plan`, gains `K_plan` around `X_plan`) from the measured state, *i.e.* `forward_pass` with $\alpha = 0$.

**Question 10 (bonus).** Implement steps 2 to 4 of the MPC loop: run `ilqr` from `U_init` with `max_iter=n_iter_mpc`, take the first control of the *new* plan `u = U_plan[0]` and apply it by advancing the state (`x = step(x, u)`), and shift the plan `X_plan`, `U_plan` **and** `K_plan` with the function `shift`.

In [ ]:
H = 40           # MPC horizon (2 s)
n_iter_mpc = 3   # iLQR iterations per control step
n_steps = 160    # 8 s of simulation
k_push_mpc = 100
push_mpc = np.array([0.0, 0.0, 0.0, 5.0])  # try this push with the feedback controller of Section 7.1!


def shift(Z):
    '''(z_0, ..., z_n) -> (z_1, ..., z_n, z_n)'''
    return np.concatenate([Z[1:], Z[-1:]])


x = x0_swing.copy()
X_plan, U_plan, K_plan, _ = ilqr(x, np.zeros((H, nu)), cost_swing)  # first solve, to convergence
X_mpc, U_mpc = [x], []

t0 = time.time()
for k in range(n_steps):
    if k == k_push_mpc:
        x = x + push_mpc
    # 1. Warm start (given): roll out the previous (shifted) feedback policy from the measured state.
    #    Only U_init is needed: ilqr recomputes the rollout itself.
    with np.errstate(over="ignore", invalid="ignore"):
        X_init, U_init = forward_pass(x, X_plan, U_plan, np.zeros((H, nu)), K_plan, alpha=0.0)
    if not np.isfinite(X_init).all():  # the policy rollout diverged: fall back on the open-loop guess
        U_init = U_plan
    # ------------------------- TODO STUDENT (begin) -------------------------
    # TODO: a few iLQR iterations from the current state, warm-started with U_init
    # TODO: apply the first control u of the new plan, by advancing the state x
    u = None
    # TODO: shift the plan (X_plan, U_plan, K_plan) for the next warm start
    # -------------------------- TODO STUDENT (end) --------------------------
    X_mpc.append(x)
    U_mpc.append(u)
X_mpc, U_mpc = np.array(X_mpc), np.array(U_mpc)
print(f"MPC: {1e3 * (time.time() - t0) / n_steps:.1f} ms per control step (real time requires < {1e3 * dt:.0f} ms)")

plot_trajectory(X_mpc, dt, U_mpc, title="MPC swing-up with a push at t = 5 s")

In [ ]:
play_trajectory(viz, X_mpc, dt)

Some questions to explore:
- What happens with a shorter horizon ($H = 20$)? With a single iLQR iteration per step?
- Replace the policy warm start by the open-loop one (`U_init = U_plan`). What happens after the push? Why?
- Increase the push. Up to which value does the MPC recover? Compare with the feedback controller of Section 7.1.
- Simulate the "real" system with a different model while the MPC keeps planning with the nominal one: build `robot_real = build_cartpole(pole_mass=0.4)` and a function `step_real` (a copy of `step` using `robot_real.model` and `robot_real.data`) used only to advance `x`. Is the MPC robust to this model error?

## To go further

- **DDP.** iLQR neglects the second-order derivatives of the dynamics $f$ in $Q_{xx}, Q_{uu}, Q_{ux}$ (a *Gauss-Newton* approximation). Keeping them gives *Differential Dynamic Programming* [1], with better convergence close to the solution, but $Q_{uu}$ may then be indefinite and must be regularized ($Q_{uu} + \mu I$).
- **Constraints.** Real actuators are bounded ($|u| \le u_{\max}$). iLQR can handle box constraints on $u$ by solving a small QP in the backward pass (*box-DDP*), and general constraints with augmented Lagrangian methods.
- **Multiple shooting.** Instead of eliminating the states, keep them as variables with the dynamics as constraints. This is better conditioned for unstable systems and allows infeasible initial guesses (*e.g.* FDDP).
- **Software.** [Crocoddyl](https://github.com/loco-3d/crocoddyl) and [Aligator](https://github.com/Simple-Robotics/aligator) are efficient C++ implementations of DDP-like solvers built on top of Pinocchio, used for legged robots and manipulators.

## References

[1] D. Mayne. A second-order gradient method for determining optimal trajectories of non-linear discrete-time systems. *International Journal of Control*, 1966.

[2] W. Li and E. Todorov. Iterative linear quadratic regulator design for nonlinear biological movement systems. *ICINCO*, 2004.

[3] Y. Tassa, T. Erez and E. Todorov. Synthesis and stabilization of complex behaviors through online trajectory optimization. *IROS*, 2012.

[4] C. Mastalli et al. Crocoddyl: An efficient and versatile framework for multi-contact optimal control. *ICRA*, 2020.

[5] R. Tedrake. *Underactuated Robotics*, course notes, MIT, [underactuated.mit.edu](https://underactuated.mit.edu).